# Speech-to-speech LLM: Kaggle / Colab runner

Runs the pipeline from the README step by step. **Kaggle:** Settings → Accelerator = GPU (T4 ×2 or P100), Internet = on.
**Colab:** Runtime → Change runtime type → T4 GPU.

Start with `configs/small.yaml`. Every training stage writes checkpoints; if the session dies, re-run the stage
with `--set train_...init_from=<checkpoint dir>`.

In [ ]:
!nvidia-smi

**Colab only (optional):** keep data/checkpoints on Google Drive so they survive disconnects.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# DATA = '/content/drive/MyDrive/s2s/data'; CKPT = '/content/drive/MyDrive/s2s/checkpoints'
DATA, CKPT = 'data', 'checkpoints'

In [ ]:
!git clone -b claude/new-session-02rye0 https://github.com/RaghunathDewal/Voice-to-voice-model-training.git
%cd Voice-to-voice-model-training
!pip install -q -e ".[dev]"
!pip install -q "kokoro>=0.9" && apt-get -qq install -y espeak-ng > /dev/null

In [ ]:
CFG = f"--config configs/small.yaml --set paths.data_dir={DATA} paths.ckpt_dir={CKPT}"
M = f"{DATA}/manifests"

## Smoke test (tiny random models, a few minutes). It must print `SMOKE TEST PASSED`.

In [ ]:
!bash scripts/smoke_test.sh outputs/smoke && pytest -q

## Step 1: LibriSpeech + Mimi latents + **Experiment 1** (CTC probe, go/no-go)

In [ ]:
!python -m s2s.prep.librispeech {CFG} --subsets dev-clean test-clean train-clean-100
for s in ["train", "dev", "test"]:
    !python -m s2s.prep.extract_mimi {CFG} --mode latents --in {M}/librispeech_{s}_raw.jsonl --out {M}/librispeech_{s}.jsonl
!python -m s2s.train.probe_ctc {CFG}

## Step 2: hotel data + **Experiment 2** (text-only tool accuracy of the base thinker)

In [ ]:
!python -m s2s.prep.hotel_data {CFG} --train 4000 --eval 200
!python -m s2s.eval.text_tools {CFG} --manifest {M}/hotel_eval_text.jsonl

## Step 3: distilled replies (behaviour alignment targets)

In [ ]:
!python -m s2s.prep.distill {CFG} --in {M}/librispeech_train.jsonl --out {M}/librispeech_train.jsonl --max-utts 20000

## Step 4: Stage 2, speech alignment

In [ ]:
!python -m s2s.train.speech_llm {CFG}

## Step 5: Stage 3, spoken tool requests

In [ ]:
!python -m s2s.prep.synth_kokoro {CFG} --in {M}/hotel_train_text.jsonl --out {M}/hotel_train_audio.jsonl --voices af_heart af_bella af_sarah am_adam am_michael
!python -m s2s.prep.synth_kokoro {CFG} --in {M}/hotel_eval_text.jsonl --out {M}/hotel_eval_audio.jsonl --voices af_nicole am_puck --seed 1
for s in ["train", "eval"]:
    !python -m s2s.prep.extract_mimi {CFG} --mode latents --in {M}/hotel_{s}_audio.jsonl --out {M}/hotel_{s}.jsonl
MIX = f"[{{path: {M}/hotel_train.jsonl, weight: 0.5}}, {{path: {M}/librispeech_train.jsonl, weight: 0.5}}]"
!python -m s2s.train.speech_llm {CFG} train_speech_llm.init_from={CKPT}/speech_llm_align train_speech_llm.output_dir={CKPT}/speech_llm_tools "train_speech_llm.train_manifests={MIX}" train_speech_llm.valid_manifest={M}/hotel_eval.jsonl train_speech_llm.max_steps=4000 train_speech_llm.warmup_steps=200

## Step 6: **Experiment 3**, speech vs text input

In [ ]:
!python -m s2s.eval.speech_llm {CFG} --speech-llm-dir {CKPT}/speech_llm_tools --manifest {M}/hotel_eval.jsonl
!python -m s2s.eval.speech_llm {CFG} --speech-llm-dir {CKPT}/speech_llm_tools --manifest {M}/librispeech_test.jsonl --max 300

## Step 7: merge LoRA, talker data, Stage 4 talker (single voice: LJSpeech here)

In [ ]:
!python -m s2s.prep.merge_lora {CFG} --speech-llm-dir {CKPT}/speech_llm_tools --out {CKPT}/thinker_merged
!python -m s2s.prep.ljspeech {CFG}
for s in ["train", "valid"]:
    !python -m s2s.prep.extract_mimi {CFG} --mode codes --in {M}/talker_{s}_raw.jsonl --out {M}/talker_{s}.jsonl
!python -m s2s.train.talker {CFG} train_talker.thinker_dir={CKPT}/thinker_merged

Listen to the samples written during training:

In [ ]:
import glob, IPython.display as ipd
for f in sorted(glob.glob(f"{CKPT}/talker/samples/*/*.wav"))[-3:]:
    print(f); ipd.display(ipd.Audio(f))

## Step 8: **Experiment 4**, talker intelligibility (Whisper WER)

In [ ]:
!python -m s2s.eval.talker {CFG} --talker-dir {CKPT}/talker --manifest {M}/talker_valid.jsonl --max 50

## Step 9: talk to it + **Experiment 5**, latency

In [ ]:
!python -m s2s.cli.chat {CFG} runtime.speech_llm_dir={CKPT}/speech_llm_tools runtime.talker_dir={CKPT}/talker --wav {DATA}/tts/hotel_eval_audio/hotel_eval_2_000000.wav --out-dir outputs/chat
ipd.display(ipd.Audio("outputs/chat/reply_1.wav"))
!python -m s2s.eval.latency {CFG} runtime.speech_llm_dir={CKPT}/speech_llm_tools runtime.talker_dir={CKPT}/talker --wav {DATA}/tts/hotel_eval_audio/*.wav --max 20

Microphone demo (opens a public gradio link):

In [ ]:
!pip install -q gradio
!python -m s2s.cli.gradio_app {CFG} runtime.speech_llm_dir={CKPT}/speech_llm_tools runtime.talker_dir={CKPT}/talker --share